### Importaciones de librerias.
- La libreria SQLite3 para la conexion a nuestra base de datos.
- La libreria pandas para el manejo de los datos.
- La libreria numpy para el manejo de elementos numericos.
- La libreria matplotlib y seaborn para la graficacion de los datos.
- La libreria scikit-learn para el uso y entrenamiento de modelos de machine learning
- Tambien se instalo el plugin wrangler que facilitara la visualizacion de los datos para un mejor analisis de los mismos.

In [1]:
import sqlite3

import numpy as np
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

### Conexion a la base de datos.

In [2]:
# Conexion a la base de datos.
conn = sqlite3.connect('database.sqlite')
query = "SELECT * FROM sqlite_master WHERE type = 'table';"

### Generar Data Frames

>Generar data frames a partir de las tablas existentes en la base de datos para realizar un analisis preliminar de los datos.

In [3]:
# Obtener todos los nombres de las tabas en la base de datos
tables = pd.read_sql_query(query, conn)['name'].tolist()

# Diccionario para almacernar todas las tablas en un dataframe.
df = {}

# Cargar cada tabla en un dataframe.
for table in tables:
    query = f"SELECT * FROM {table};"
    df[table] = pd.read_sql_query(query, conn)

# Cerrar la conexion a la base de datos.
conn.close()

# Mostrar los nombres de las tablas.
df.keys()


dict_keys(['sqlite_sequence', 'Player_Attributes', 'Player', 'Match', 'League', 'Country', 'Team', 'Team_Attributes'])

### Analisis Exploratorio de Datos (EDA)
> Exploracion de los datos: explorar la estructura del dataset para identificar tipos de datos y valores nulos.

In [4]:
# Mostrar informacion general sobre la tabla.
df['sqlite_sequence'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    7 non-null      object
 1   seq     7 non-null      int64 
dtypes: int64(1), object(1)
memory usage: 244.0+ bytes


In [5]:
# Mostrar informacion de datos numericos sobre la tabla.
df['sqlite_sequence'].describe()

,seq
count,7.000000
mean,65185.857143
std,62082.942398
min,1458.000000
25%,31516.500000
50%,51958.000000
75%,77937.000000
max,183978.000000


> La tabla "sqlite_sequence" cuenta solamente con 2 columnas y contiene un total de 7 registros completos sin nulos o valores atipicos en sus datos.
- cuenta con una (1) columna tipo string.
- cuenta con una (1) columna tipo entero.

In [6]:
# Mostrar informacion general sobre la tabla.
df['Player_Attributes'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 183978 entries, 0 to 183977
Data columns (total 42 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   id                   183978 non-null  int64  
 1   player_fifa_api_id   183978 non-null  int64  
 2   player_api_id        183978 non-null  int64  
 3   date                 183978 non-null  object 
 4   overall_rating       183142 non-null  float64
 5   potential            183142 non-null  float64
 6   preferred_foot       183142 non-null  object 
 7   attacking_work_rate  180748 non-null  object 
 8   defensive_work_rate  183142 non-null  object 
 9   crossing             183142 non-null  float64
 10  finishing            183142 non-null  float64
 11  heading_accuracy     183142 non-null  float64
 12  short_passing        183142 non-null  float64
 13  volleys              181265 non-null  float64
 14  dribbling            183142 non-null  float64
 15  curve            

In [7]:
# Mostrar informacion de datos numericos sobre la tabla.
df['Player_Attributes'].describe()

,id,player_fifa_api_id,player_api_id,overall_rating,potential,crossing,finishing,heading_accuracy,short_passing,volleys,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
count,183978.00000,183978.000000,183978.000000,183142.000000,183142.000000,183142.000000,183142.000000,183142.000000,183142.000000,181265.000000,...,181265.000000,183142.000000,183142.000000,183142.000000,181265.000000,183142.000000,183142.000000,183142.000000,183142.000000,183142.000000
mean,91989.50000,165671.524291,135900.617324,68.600015,73.460353,55.086883,49.921078,57.266023,62.429672,49.468436,...,57.873550,55.003986,46.772242,50.351257,48.001462,14.704393,16.063612,20.998362,16.132154,16.441439
std,53110.01825,53851.094769,136927.840510,7.041139,6.592271,17.242135,19.038705,16.488905,14.194068,18.256618,...,15.144086,15.546519,21.227667,21.483706,21.598778,16.865467,15.867382,21.452980,16.099175,17.198155
min,1.00000,2.000000,2625.000000,33.000000,39.000000,1.000000,1.000000,1.000000,3.000000,1.000000,...,1.000000,2.000000,1.000000,1.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,45995.25000,155798.000000,34763.000000,64.000000,69.000000,45.000000,34.000000,49.000000,57.000000,35.000000,...,49.000000,45.000000,25.000000,29.000000,25.000000,7.000000,8.000000,8.000000,8.000000,8.000000
50%,91989.50000,183488.000000,77741.000000,69.000000,74.000000,59.000000,53.000000,60.000000,65.000000,52.000000,...,60.000000,57.000000,50.000000,56.000000,53.000000,10.000000,11.000000,12.000000,11.000000,11.000000
75%,137983.75000,199848.000000,191080.000000,73.000000,78.000000,68.000000,65.000000,68.000000,72.000000,64.000000,...,69.000000,67.000000,66.000000,69.000000,67.000000,13.000000,15.000000,15.000000,15.000000,15.000000
max,183978.00000,234141.000000,750584.000000,94.000000,97.000000,95.000000,97.000000,98.000000,97.000000,93.000000,...,97.000000,96.000000,96.000000,95.000000,95.000000,94.000000,93.000000,97.000000,96.000000,96.000000


In [8]:
df['Player_Attributes'].isnull().sum().sort_values()

id                        0
player_fifa_api_id        0
player_api_id             0
date                      0
overall_rating          836
potential               836
preferred_foot          836
defensive_work_rate     836
short_passing           836
crossing                836
finishing               836
heading_accuracy        836
dribbling               836
ball_control            836
long_passing            836
free_kick_accuracy      836
interceptions           836
aggression              836
strength                836
acceleration            836
sprint_speed            836
reactions               836
stamina                 836
shot_power              836
positioning             836
long_shots              836
gk_handling             836
gk_kicking              836
gk_diving               836
standing_tackle         836
marking                 836
penalties               836
gk_reflexes             836
gk_positioning          836
sliding_tackle         2713
jumping             

> La tabla "Player_Attributes" cuenta con 42 columnas y 183978 registros en su totalidad, sin embargo se pueden observar diversos valores nulos en diferentes columnas, la columna que contiene fechas esta como texto, se evaluara como proceder con su tratamiento de los datos faltantes y la columna fechas.

- cuenta con treinta y cinco (35) columnas tipo Flotantes.
- cuenta con tres (3) columnas tipo entero.
- cuenta con cuatro (4) columnas tipo texto.

In [9]:
# Mostrar informacion general sobre la tabla.
df['Player'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11060 entries, 0 to 11059
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  11060 non-null  int64  
 1   player_api_id       11060 non-null  int64  
 2   player_name         11060 non-null  object 
 3   player_fifa_api_id  11060 non-null  int64  
 4   birthday            11060 non-null  object 
 5   height              11060 non-null  float64
 6   weight              11060 non-null  int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 605.0+ KB


In [10]:
# Mostrar informacion de datos numericos sobre la tabla.
df['Player'].describe()

,id,player_api_id,player_fifa_api_id,height,weight
count,11060.000000,11060.000000,11060.000000,11060.000000,11060.000000
mean,5537.511392,156582.427215,165664.910488,181.867445,168.380289
std,3197.692647,160713.700624,58649.928360,6.369201,14.990217
min,1.000000,2625.000000,2.000000,157.480000,117.000000
25%,2767.750000,35555.500000,151889.500000,177.800000,159.000000
50%,5536.500000,96619.500000,184671.000000,182.880000,168.000000
75%,8306.250000,212470.500000,203883.250000,185.420000,179.000000
max,11075.000000,750584.000000,234141.000000,208.280000,243.000000


> La tabla "Player" cuenta con 7 columnas y 11060 registros registros completos sin nulos, la columna fechas se encuentra en un formato de texto.

- cuenta con una (1) columna tipo Flotante.
- cuenta con  cuatro (4) columnas tipo entero.
- cuenta con dos (2) columnas tipo texto.

In [11]:
# Mostrar informacion general sobre la tabla.
df['Match'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25979 entries, 0 to 25978
Columns: 115 entries, id to BSA
dtypes: float64(96), int64(9), object(10)
memory usage: 22.8+ MB


In [12]:
# Mostrar informacion de datos numericos sobre la tabla.
df['Match'].describe()

,id,country_id,league_id,stage,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,away_team_goal,home_player_X1,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
count,25979.000000,25979.000000,25979.000000,25979.000000,2.597900e+04,25979.000000,25979.000000,25979.000000,25979.000000,24158.000000,...,17097.000000,22568.000000,22568.000000,22568.000000,14162.000000,14162.000000,14162.000000,14161.000000,14161.000000,14161.000000
mean,12990.000000,11738.630317,11738.630317,18.242773,1.195429e+06,9984.371993,9984.475115,1.544594,1.160938,0.999586,...,4.622343,2.668107,3.899048,4.840281,2.498764,3.648189,4.353097,2.497894,3.660742,4.405663
std,7499.635658,7553.936759,7553.936759,10.407354,4.946279e+05,14087.453758,14087.445135,1.297158,1.142110,0.022284,...,3.632164,1.928753,1.248221,4.318338,1.489299,0.867440,3.010189,1.507793,0.868272,3.189814
min,1.000000,1.000000,1.000000,1.000000,4.831290e+05,1601.000000,1601.000000,0.000000,0.000000,0.000000,...,1.100000,1.030000,1.620000,1.080000,1.050000,1.450000,1.120000,1.040000,1.330000,1.120000
25%,6495.500000,4769.000000,4769.000000,9.000000,7.684365e+05,8475.000000,8475.000000,1.000000,0.000000,1.000000,...,2.500000,1.700000,3.300000,2.550000,1.670000,3.200000,2.500000,1.670000,3.250000,2.500000
50%,12990.000000,10257.000000,10257.000000,18.000000,1.147511e+06,8697.000000,8697.000000,1.000000,1.000000,1.000000,...,3.500000,2.150000,3.500000,3.500000,2.100000,3.300000,3.400000,2.100000,3.400000,3.400000
75%,19484.500000,17642.000000,17642.000000,27.000000,1.709852e+06,9925.000000,9925.000000,2.000000,2.000000,1.000000,...,5.250000,2.800000,4.000000,5.400000,2.650000,3.750000,5.000000,2.620000,3.750000,5.000000
max,25979.000000,24558.000000,24558.000000,38.000000,2.216672e+06,274581.000000,274581.000000,10.000000,9.000000,2.000000,...,41.000000,36.000000,26.000000,67.000000,21.000000,11.000000,34.000000,17.000000,13.000000,34.000000


> La tabla "Match" cuenta con 115 columnas y 25979 registros en su totalidad, sin embargo se pueden observar diversos valores nulos en diferentes columnas, existen dos columnas que contienen fechas y se encuentran en formato de tipo texto, tambien se observario registros atipicos 8 de las columnas, se evaluara como proceder con su tratamiento de los datos faltantes, valores atipicos y columnas fechas.


- cuenta con noventa y seis (96) columna tipo Flotante.
- cuenta con  nueve (9) columnas tipo entero.
- cuenta con diez (10) columnas tipo texto.

In [13]:
# Mostrar informacion general sobre la tabla.
df['League'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          11 non-null     int64 
 1   country_id  11 non-null     int64 
 2   name        11 non-null     object
dtypes: int64(2), object(1)
memory usage: 396.0+ bytes


In [14]:
# Mostrar informacion de datos numericos sobre la tabla.
df['League'].describe()

,id,country_id
count,11.000000,11.000000
mean,12452.090909,12452.090909
std,8215.308472,8215.308472
min,1.000000,1.000000
25%,6289.000000,6289.000000
50%,13274.000000,13274.000000
75%,18668.000000,18668.000000
max,24558.000000,24558.000000


> La tabla "League" cuenta con 3 columnas y 11 registros en su totalidad, sin nulos.



- cuenta con  dos (2) columnas tipo entero.
- cuenta con una (1) columna tipo texto.

In [15]:
# Mostrar informacion general sobre la tabla.
df['Country'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      11 non-null     int64 
 1   name    11 non-null     object
dtypes: int64(1), object(1)
memory usage: 308.0+ bytes


In [16]:
# Mostrar informacion de datos numericos sobre la tabla.
df['Country'].describe()

,id
count,11.000000
mean,12452.090909
std,8215.308472
min,1.000000
25%,6289.000000
50%,13274.000000
75%,18668.000000
max,24558.000000


> La tabla "Country" cuenta solamente con 2 columnas y contiene un total de 11 registros completos sin nulos o valores atipicos en sus datos.
- cuenta con una (1) columna tipo string.
- cuenta con una (1) columna tipo entero.

In [17]:
# Mostrar informacion general sobre la tabla.
df['Team'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                299 non-null    int64  
 1   team_api_id       299 non-null    int64  
 2   team_fifa_api_id  288 non-null    float64
 3   team_long_name    299 non-null    object 
 4   team_short_name   299 non-null    object 
dtypes: float64(1), int64(2), object(2)
memory usage: 11.8+ KB


In [18]:
# Mostrar informacion de datos numericos sobre la tabla.
df['Team'].describe()

,id,team_api_id,team_fifa_api_id
count,299.000000,299.000000,288.000000
mean,23735.301003,12340.521739,21534.305556
std,15167.914719,25940.411135,42456.439408
min,1.000000,1601.000000,1.000000
25%,9552.500000,8349.000000,178.750000
50%,22805.000000,8655.000000,673.500000
75%,36250.500000,9886.500000,1910.750000
max,51606.000000,274581.000000,112513.000000


> La tabla "Team" cuenta solamente con 5 columnas y contiene un total de 299 registros, tiene pocos valores nulos y sin valores atipicos en sus datos.
- cuenta con dos (2) columnas tipo string.
- cuenta con dos (2) columnas tipo entero.
- cuenta con una (1) columna tipo Flotante.

In [19]:
# Mostrar informacion general sobre la tabla.
df['Team_Attributes'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1458 entries, 0 to 1457
Data columns (total 25 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              1458 non-null   int64  
 1   team_fifa_api_id                1458 non-null   int64  
 2   team_api_id                     1458 non-null   int64  
 3   date                            1458 non-null   object 
 4   buildUpPlaySpeed                1458 non-null   int64  
 5   buildUpPlaySpeedClass           1458 non-null   object 
 6   buildUpPlayDribbling            489 non-null    float64
 7   buildUpPlayDribblingClass       1458 non-null   object 
 8   buildUpPlayPassing              1458 non-null   int64  
 9   buildUpPlayPassingClass         1458 non-null   object 
 10  buildUpPlayPositioningClass     1458 non-null   object 
 11  chanceCreationPassing           1458 non-null   int64  
 12  chanceCreationPassingClass      14

In [20]:
# Mostrar informacion de datos numericos sobre la tabla.
df['Team_Attributes'].describe()

,id,team_fifa_api_id,team_api_id,buildUpPlaySpeed,buildUpPlayDribbling,buildUpPlayPassing,chanceCreationPassing,chanceCreationCrossing,chanceCreationShooting,defencePressure,defenceAggression,defenceTeamWidth
count,1458.000000,1458.000000,1458.000000,1458.000000,489.000000,1458.000000,1458.000000,1458.000000,1458.000000,1458.000000,1458.000000,1458.000000
mean,729.500000,17706.982167,9995.727023,52.462277,48.607362,48.490398,52.165295,53.731824,53.969136,46.017147,49.251029,52.185871
std,421.032659,39179.857739,13264.869900,11.545869,9.678290,10.896101,10.360793,11.086796,10.327566,10.227225,9.738028,9.574712
min,1.000000,1.000000,1601.000000,20.000000,24.000000,20.000000,21.000000,20.000000,22.000000,23.000000,24.000000,29.000000
25%,365.250000,110.000000,8457.750000,45.000000,42.000000,40.000000,46.000000,47.000000,48.000000,39.000000,44.000000,47.000000
50%,729.500000,485.000000,8674.000000,52.000000,49.000000,50.000000,52.000000,53.000000,53.000000,45.000000,48.000000,52.000000
75%,1093.750000,1900.000000,9904.000000,62.000000,55.000000,55.000000,59.000000,62.000000,61.000000,51.000000,55.000000,58.000000
max,1458.000000,112513.000000,274581.000000,80.000000,77.000000,80.000000,80.000000,80.000000,80.000000,72.000000,72.000000,73.000000


> La tabla "Team_Attributes" cuenta con 25 columnas y 1458 registros en su totalidad, sin embargo se pueden observar diversos valores nulos en diferentes columnas, se evaluara como proceder con su tratamiento de los datos faltantes, valores atipicos y columnas fechas.


- cuenta con una (1) columna tipo Flotante.
- cuenta con  once (11) columnas tipo entero.
- cuenta con trece (13) columnas tipo texto.

## Transformacion y carga de los datos para disponibilizarlos ETL

In [21]:
# Nombre de columna a tratar.
column = 'defensive_work_rate'

# Mostrar los registros que contengan valores nulos en la tabla.
df_null = df['Player_Attributes'][df['Player_Attributes'].isnull().any(axis=1)]
df_null

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
373,374,156626,46447,2010-08-30 00:00:00,64.0,71.0,right,None,_0,41.0,...,61.0,39.0,62.0,61.0,57.0,15.0,14.0,13.0,10.0,12.0
374,375,156626,46447,2010-02-22 00:00:00,64.0,71.0,right,None,_0,41.0,...,61.0,58.0,62.0,61.0,57.0,6.0,20.0,45.0,20.0,20.0
375,376,156626,46447,2008-08-30 00:00:00,66.0,71.0,right,None,_0,41.0,...,61.0,58.0,67.0,61.0,57.0,6.0,20.0,45.0,20.0,20.0
376,377,156626,46447,2007-08-30 00:00:00,68.0,75.0,right,None,_0,41.0,...,61.0,58.0,69.0,64.0,57.0,6.0,20.0,45.0,20.0,20.0
377,378,156626,46447,2007-02-22 00:00:00,66.0,65.0,right,None,_0,41.0,...,61.0,55.0,66.0,63.0,57.0,6.0,9.0,45.0,13.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183963,183964,47058,35506,2009-08-30 00:00:00,70.0,78.0,right,None,_0,48.0,...,63.0,70.0,70.0,74.0,65.0,14.0,25.0,51.0,25.0,25.0
183964,183965,47058,35506,2009-02-22 00:00:00,70.0,78.0,right,None,_0,48.0,...,63.0,70.0,70.0,74.0,65.0,14.0,25.0,51.0,25.0,25.0
183965,183966,47058,35506,2008-08-30 00:00:00,72.0,78.0,right,None,_0,48.0,...,63.0,70.0,76.0,78.0,65.0,14.0,25.0,51.0,25.0,25.0
183966,183967,47058,35506,2007-08-30 00:00:00,75.0,78.0,right,None,_0,48.0,...,63.0,70.0,76.0,78.0,65.0,14.0,25.0,51.0,25.0,25.0


In [22]:
# Mostrar los valores unicos en la columna.
df['Player_Attributes'][column].unique()

array(['medium', 'high', 'low', '_0', None, '5', 'ean', 'o', '1', 'ormal',
       '7', '2', '8', '4', 'tocky', '0', '3', '6', '9', 'es'],
      dtype=object)

In [23]:
df['Player_Attributes']

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21 00:00:00,62.0,66.0,right,medium,medium,49.0,...,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183973,183974,102359,39902,2009-08-30 00:00:00,83.0,85.0,right,medium,low,84.0,...,88.0,83.0,22.0,31.0,30.0,9.0,20.0,84.0,20.0,20.0
183974,183975,102359,39902,2009-02-22 00:00:00,78.0,80.0,right,medium,low,74.0,...,88.0,70.0,32.0,31.0,30.0,9.0,20.0,73.0,20.0,20.0
183975,183976,102359,39902,2008-08-30 00:00:00,77.0,80.0,right,medium,low,74.0,...,88.0,70.0,32.0,31.0,30.0,9.0,20.0,73.0,20.0,20.0
183976,183977,102359,39902,2007-08-30 00:00:00,78.0,81.0,right,medium,low,74.0,...,88.0,53.0,28.0,32.0,30.0,9.0,20.0,73.0,20.0,20.0


In [24]:
# Evaluar la moda de la columna tratada.
mode_value = df['Player_Attributes'][column].mode()[0]

# Valores a Reemplazar.
replaces = {
    "0":"low","_0":"low", "o":"low","1":"low","2":"low","3":"low",
            "4":"medium","5":"medium","6":"medium",
            "7":"medium","8":"high","9":"high", 
            "ormal":f"{mode_value}", "ean":f"{mode_value}","tocky": f"{mode_value}", 
            "es":f"{mode_value}"
            }

# Reemplazar valores nulos por la moda.
df['Player_Attributes'][column] = df['Player_Attributes'][column].fillna(mode_value)
# Reemplazar valores numericos por low, medium y high.
df['Player_Attributes'][column] = df['Player_Attributes'][column].replace(replaces) 

df['Player_Attributes']



,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21 00:00:00,62.0,66.0,right,medium,medium,49.0,...,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183973,183974,102359,39902,2009-08-30 00:00:00,83.0,85.0,right,medium,low,84.0,...,88.0,83.0,22.0,31.0,30.0,9.0,20.0,84.0,20.0,20.0
183974,183975,102359,39902,2009-02-22 00:00:00,78.0,80.0,right,medium,low,74.0,...,88.0,70.0,32.0,31.0,30.0,9.0,20.0,73.0,20.0,20.0
183975,183976,102359,39902,2008-08-30 00:00:00,77.0,80.0,right,medium,low,74.0,...,88.0,70.0,32.0,31.0,30.0,9.0,20.0,73.0,20.0,20.0
183976,183977,102359,39902,2007-08-30 00:00:00,78.0,81.0,right,medium,low,74.0,...,88.0,53.0,28.0,32.0,30.0,9.0,20.0,73.0,20.0,20.0


In [25]:

df['Player_Attributes'][column].unique()

array(['medium', 'high', 'low'], dtype=object)

### Consideraciones:
> Se decide dejar rellenar los valores nulos en lugar de reemplazarlos ya que cada valor representa un jugador unico y se de eliminar los valores nulos se estaria perdiendo la informacion general de ese jugador en particular, por esa razon se conservan dejando en los valores con texto con la moda de los valores totales en cada columna.